<a href="https://colab.research.google.com/github/Nickhilsekar/spider_ml/blob/main/spider_basic_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Pipeline and Data Loading
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 2. Model Architecture
class ConvolutionalAutoencoder(nn.Module):
    def __init__(self):
        super(ConvolutionalAutoencoder, self).__init__()

        # Encoder: Downsamples from 28x28 to 1x1
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # Out: 16 x 14 x 14
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # Out: 32 x 7 x 7
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=7)                       # Out: 64 x 1 x 1
        )

        # Decoder: Reconstructs back to 28x28
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=7),             # Out: 32 x 7 x 7
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # Out: 16 x 14 x 14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),  # Out: 1 x 28 x 28
            nn.Sigmoid()                                           # Bounds pixels between 0 and 1
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# 3. Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ConvolutionalAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Helper function to compute structural similarity / reconstruction accuracy
def calculate_accuracy(original, reconstructed, tolerance=0.1):
    """
    Calculates accuracy based on the percentage of pixels reconstructed
    within an acceptable intensity threshold (tolerance).
    """
    diff = torch.abs(original - reconstructed)
    correct_pixels = torch.sum(diff < tolerance).item()
    total_pixels = original.numel()
    return (correct_pixels / total_pixels) * 100

# 4. Training and Evaluation Loop
num_epochs = 5
print(f"Starting training on device: {device}\n" + "-"*40)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_accuracy = 0.0

    for images, _ in train_loader:
        images = images.to(device)

        # Optimization pass
        outputs = model(images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        train_accuracy += calculate_accuracy(images, outputs) * images.size(0)

    # Validation phase
    model.eval()
    test_loss = 0.0
    test_accuracy = 0.0

    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            outputs = model(images)

            loss = criterion(outputs, images)
            test_loss += loss.item() * images.size(0)
            test_accuracy += calculate_accuracy(images, outputs) * images.size(0)

    # Compute epoch metrics
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = train_accuracy / len(train_loader.dataset)
    epoch_test_loss = test_loss / len(test_loader.dataset)
    epoch_test_acc = test_accuracy / len(test_loader.dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train Loss: {epoch_train_loss:.4f} | Train Pixel Accuracy: {epoch_train_acc:.2f}%")
    print(f"  Test Loss:  {epoch_test_loss:.4f} | Test Pixel Accuracy:  {epoch_test_acc:.2f}%")
    print("-"*40)


100%|██████████| 26.4M/26.4M [00:01<00:00, 21.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 348kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.59MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.2MB/s]


Starting training on device: cpu
----------------------------------------
Epoch [1/5]
  Train Loss: 0.0297 | Train Pixel Accuracy: 67.90%
  Test Loss:  0.0142 | Test Pixel Accuracy:  77.95%
----------------------------------------
Epoch [2/5]
  Train Loss: 0.0117 | Train Pixel Accuracy: 80.77%
  Test Loss:  0.0103 | Test Pixel Accuracy:  82.24%
----------------------------------------
Epoch [3/5]
  Train Loss: 0.0095 | Train Pixel Accuracy: 83.31%
  Test Loss:  0.0090 | Test Pixel Accuracy:  83.85%
----------------------------------------
Epoch [4/5]
  Train Loss: 0.0084 | Train Pixel Accuracy: 84.69%
  Test Loss:  0.0081 | Test Pixel Accuracy:  84.88%
----------------------------------------
Epoch [5/5]
  Train Loss: 0.0077 | Train Pixel Accuracy: 85.60%
  Test Loss:  0.0075 | Test Pixel Accuracy:  85.75%
----------------------------------------
